In [1]:
import os
from glob import glob

import numpy as np
from ultralytics import YOLO

In [2]:
model = YOLO("../model/checkpoints/yolo11n-obb-best.pt")

In [ ]:
results = model.val(
    source="../data/combined_wildlife_poly_bridges_filtered_no_ch_640-yolo/images/test",
    project="cassda-wildlife-crossing",
    name="yolo11-CH-conf0_35-test",
    exist_ok=True,
    save=True,
    plots=True,
)

Ultralytics 8.3.40  Python-3.13.1 torch-2.9.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti SUPER, 16376MiB)


val: Scanning C:\code\cassda-zertifikatsarbeit\data\combined_wildlife_poly_bridges_filtered_no_ch_640-yolo\labels\val.cache... 82 images, 12 backgrounds, 0 corrupt: 100%|██████████| 82/82 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:04<00:00,  1.25it/s]


                   all         82         70      0.912      0.886      0.929      0.531
Speed: 2.2ms preprocess, 12.1ms inference, 0.0ms loss, 3.2ms postprocess per image
Results saved to cassda-wildlife-crossing\yolo11-CH-conf0_35-test


: 

In [ ]:
results = model.predict(
    source="../data/combined_wildlife_poly_bridges_filtered_no_ch_640-yolo/images/test",
    conf=0.35,
    project="cassda-wildlife-crossing",
    name="yolo11-CH-conf0_35-test",
    exist_ok=True,
    save_json=True,
    save_txt=True,
    visualize=False,
)


image 1/84 c:\code\cassda-zertifikatsarbeit\notebooks\..\data\combined_wildlife_poly_bridges_filtered_no_ch_640-yolo\images\test\4024916_2954868_4026196_2956148.jpg: 640x640 (no detections), 24.0ms
image 2/84 c:\code\cassda-zertifikatsarbeit\notebooks\..\data\combined_wildlife_poly_bridges_filtered_no_ch_640-yolo\images\test\4031099_2903626_4032379_2904906.jpg: 640x640 (no detections), 17.6ms
image 3/84 c:\code\cassda-zertifikatsarbeit\notebooks\..\data\combined_wildlife_poly_bridges_filtered_no_ch_640-yolo\images\test\4035118_2849877_4036398_2851157.jpg: 640x640 (no detections), 16.9ms
image 4/84 c:\code\cassda-zertifikatsarbeit\notebooks\..\data\combined_wildlife_poly_bridges_filtered_no_ch_640-yolo\images\test\4035249_3148429_4036529_3149709.jpg: 640x640 (no detections), 23.1ms
image 5/84 c:\code\cassda-zertifikatsarbeit\notebooks\..\data\combined_wildlife_poly_bridges_filtered_no_ch_640-yolo\images\test\4038236_2929307_4039516_2930587.jpg: 640x640 (no detections), 19.5ms
image 6/8

In [4]:
gt_dir = "../data/combined_wildlife_poly_bridges_filtered_no_ch_640-yolo/labels/test"
pred_dir = "../model/runs/cassda-wildlife-crossing/yolo11-CH-conf0_35-test/labels"

output_dir = (
    "../notebooks/cassda-wildlife-crossing/yolo11-CH-conf0_35-test/box-visualizations"
)

In [5]:
os.makedirs(output_dir, exist_ok=True)

In [10]:
import cv2
import matplotlib.pyplot as plt

for result in results:
    img_name = os.path.basename(result.path)
    print(f"Processing {img_name}...")
    base_name = os.path.splitext(img_name)[0]

    gt_path = os.path.join(gt_dir, f"{base_name}.txt")
    pred_path = os.path.join(pred_dir, f"{base_name}.txt")

    gt_polygons = []
    if os.path.exists(gt_path):
        with open(gt_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                cls_id = int(parts[0])
                bbox = list(map(float, parts[1:]))
                gt_polygons.append((cls_id, bbox))

    pred_boxes = []
    result_boxes = result.obb

    if result_boxes is None and gt_polygons == []:
        continue
    for box in result_boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        bbox = box.xyxyxyxy[0].tolist()
        pred_boxes.append((cls_id, conf, bbox))

    img = cv2.imread(result.path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    for cls_id, polygon in gt_polygons:
        pts = [p * 640 for p in polygon]
        pts = np.array(pts, dtype=np.int32).reshape((-1, 1, 2))

        cv2.polylines(
            img,
            [pts],
            isClosed=True,
            color=(0, 0, 255),
            thickness=1,
        )
        cv2.putText(
            img,
            "GT",
            tuple(pts[0][0] + np.array([0, -15])),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 0, 255),
            2,
        )

    for cls_id, conf, obb in pred_boxes:
        obb = [p * 640 for p in obb]
        pts = np.array(obb, dtype=np.int32).reshape((-1, 1, 2))
        cv2.polylines(
            img,
            [pts],
            isClosed=True,
            color=(255, 0, 0),
            thickness=1,
        )
        cv2.putText(
            img,
            f"Pred. ({conf:.2f})",
            tuple(pts[0][0] + np.array([0, 15])),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255, 0, 0),
            2,
        )

    output_path = os.path.join(output_dir, img_name)

    # plt.figure(figsize=(10, 10))
    # plt.imshow(img)
    # plt.axis("off")
    # plt.title(f"Image: {img_name}")
    # plt.show()
    cv2.imwrite(output_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    if len(gt_polygons) > len(pred_boxes):
        cv2.imwrite(
            os.path.join(output_dir, "false-negative", img_name),
            cv2.cvtColor(img, cv2.COLOR_RGB2BGR),
        )
        print(
            f"Saved false negative to {os.path.join(output_dir, 'false-negative', img_name)}"
        )

Processing 4024916_2954868_4026196_2956148.jpg...
Processing 4031099_2903626_4032379_2904906.jpg...
Processing 4035118_2849877_4036398_2851157.jpg...
Processing 4035249_3148429_4036529_3149709.jpg...
Processing 4038236_2929307_4039516_2930587.jpg...
Processing 4039090_2894888_4040370_2896168.jpg...
Processing 4040309_2946224_4041589_2947504.jpg...
Processing 4041706_2905360_4042986_2906640.jpg...
Processing 4041938_3170740_4043218_3172020.jpg...
Processing 4042840_3165992_4044120_3167272.jpg...
Processing 4047113_2900638_4048393_2901918.jpg...
Processing 4048657_3108163_4049937_3109443.jpg...
Processing crossing_100.jpg...
Processing crossing_104.jpg...
Processing crossing_111.jpg...
Saved false negative to ../notebooks/cassda-wildlife-crossing/yolo11-CH-conf0_35-test/box-visualizations\false-negative\crossing_111.jpg
Processing crossing_114.jpg...
Processing crossing_117.jpg...
Processing crossing_118.jpg...
Saved false negative to ../notebooks/cassda-wildlife-crossing/yolo11-CH-conf0